In [1]:
# - play with the 7k dataset to understand the image dimensions and the bbox dimensions
# - all images in this dataset are resized to 840 x 840. the bboxes are in the resized dimensions
# - in our final dataset, the image needs to be resized to resize_size (1024) and the bboxes should be resized to prediction_grid_size (1000)
# -- divide by 840 and multiply by 1000 to get the final bboxes and points
# -- for sam3 baseline box prediction, we can just use the resized image of 840 x 840 and the gt boxes listed in the solution 

In [2]:
# changes from segzero dataset creation:
# 1. qwen3 outputs all boxes in 1000x1000 resolution, so the ground truth "resize_size" needs to be 1000x1000
# 2. qwen3 vision encoder actually expects images to be multple of 32, so we will resize images to 102x1024
# 3. we will add the sam3 boxes iou as a dataset column "baseline_iou"
# 4. we don't need "object_hint_boxes" since we assume all queries are for objects in the segzero dataset
# -- we can verify this using chatgpt as an additional step of data curation if needed 
## if all 4 components of a box are 0, we will set that box to None instead of [0,0,0,0]. This needs to be handled during training

In [3]:
# format in segzero format 
from datasets import Dataset, DatasetDict, Image, Features, Value
import json
from tqdm import tqdm
from tqdm import tqdm
import cv2

/home/ksmehrab/miniconda/envs/segzero/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from datasets import load_dataset
data_path = "/data/VLMGroundingProject/Datasets/SegZeroVisualReasoner/VisionReasoner_multi_object_7k_840"
dataset = load_dataset(data_path)
dataset = dataset['train']

In [5]:
len(dataset)

7099

In [6]:
dataset[0]

{'id': 'refcocog_28421',
 'problem': "'this is a display screen for presentations, showing exactly what the computer screen on the table'",
 'solution': '[{"bbox_2d": [35, 74, 585, 500], "point_2d": [295, 284]}]',
 'image': <PIL.PngImagePlugin.PngImageFile image mode=RGB size=840x840>,
 'img_height': 640,
 'img_width': 478}

#### Old code

In [7]:
from PIL import Image as PILImage
def scale_box_coordinates(bbox_2d, x_factor, y_factor):
    """
    bbox_2d: [x1, y1, x2, y2]
    """

    scaled_bbox = [
        int(bbox_2d[0] * x_factor + 0.5),  # x1
        int(bbox_2d[1] * y_factor + 0.5),  # y1
        int(bbox_2d[2] * x_factor + 0.5),  # x2
        int(bbox_2d[3] * y_factor + 0.5)   # y2
    ]
    
    return scaled_bbox

def scale_point_coordinates(point_2d, x_factor, y_factor):
    """
    point_2d: [x, y]
    """
    scaled_point = [
        int(point_2d[0] * x_factor + 0.5),  # x
        int(point_2d[1] * y_factor + 0.5)   # y
    ]
    
    return scaled_point

def create_local_dataset(train_data, output_dir, image_resize):
    def process_split(split_data, image_resize):
        processed_data = split_data.copy()
        images = []
        for img in split_data['image']:
            image_resized = img.resize((image_resize, image_resize), PILImage.Resampling.LANCZOS)
            images.append(image_resized)
        
        processed_data['image'] = images
        return processed_data
    
    dataset = DatasetDict({
        'train': Dataset.from_dict(
            process_split(train_data, image_resize),
            features=Features({
                'id': Value('string'),
                'problem': Value('string'),
                'solution': Value('string'),
                'image': Image(),
                'img_height': Value('int64'),
                'img_width': Value('int64'),
                'object_part': Value('bool'),
                'object_hint_boxes': Value('string'),
                'baseline_iou': Value('float32'),
            })
        )
    })
    
    dataset.save_to_disk(output_dir)
    print(f"saved to: {output_dir}")
    
    return dataset

In [8]:
image_resize = 1024
prediction_grid_size = 1000
    
id_list = []
problem_list = []
solution_list = []
image_list = []
img_height_list = []
img_width_list = []
# create an object_part_list. add False for all entries since this is the segzero dataset and we assume all queries are for objects
object_part_list = [False] * len(dataset)
# create an object_hint_boxes_list. add None for all entries since this is the segzero dataset and we assume all queries are for objects
# in the reward training, firstly, if it is an object part, it will return 0 reward right away. so no need to provide any boxes. If it gets None, it will eventually return 0 reward. 
# this creates a discrepancy between total part rewards and total object rewards, which we handle via normalization by the max achievable reward
object_boxes_list = [None] * len(dataset)
baseline_iou_list = []

In [9]:
# read sam3 baseline boxes data 
sam3_boxes_data = []
for i in range(4):
    sam3_boxes_data_path = f"/data/VLMGroundingProject/Datasets/SegZeroVisualReasoner/Sam3_Boxes/sam3_bboxes_{i}.json"
    sam3_boxes_data.extend(json.load(open(sam3_boxes_data_path, 'r')))

In [10]:
image_name_to_sam3_boxes = {}
for item in sam3_boxes_data:
    image_name = item['image_name']
    sam3_bboxes = item['pred_bboxes']  
    if sam3_bboxes == [0,0,0,0]:
        sam3_bboxes = None
    image_name_to_sam3_boxes[image_name] = sam3_bboxes

In [11]:
# compute baseline iou (sam3 boxes vs gt boxes) and add it to the dataset right here. so we don't have to compute baseline iou during training

import numpy as np
from scipy.optimize import linear_sum_assignment

def batch_iou(boxes1, boxes2):
    """Compute IoU between each box in boxes1 and each box in boxes2."""
    x11, y11, x12, y12 = np.split(boxes1, 4, axis=1)  # (M,1) each
    x21, y21, x22, y22 = np.split(boxes2, 4, axis=1)  # (N,1)
    # Intersection coords
    xA = np.maximum(x11, np.transpose(x21))
    yA = np.maximum(y11, np.transpose(y21))
    xB = np.minimum(x12, np.transpose(x22))
    yB = np.minimum(y12, np.transpose(y22))
    interArea = np.maximum(0, xB - xA + 1) * np.maximum(0, yB - yA + 1)
    # Areas of boxes
    box1Area = (x12 - x11 + 1) * (y12 - y11 + 1)
    box2Area = (x22 - x21 + 1) * (y22 - y21 + 1)
    # Union
    unionArea = box1Area + np.transpose(box2Area) - interArea
    iou = interArea / np.clip(unionArea, a_min=1e-9, a_max=None)  # avoid division by zero
    return iou

def compute_hungarian_match_and_average_iou(pred_bboxes: np.ndarray, gt_bboxes: np.ndarray) -> float:
    try:
        M, N = len(pred_bboxes), len(gt_bboxes)
        if M == 0 or N == 0:
            # if both empty, full reward
            if M == 0 and N == 0:
                return 1.0  # no objects quried and none predicted, so its correct.
            return 0.0  # no initial prediction or no target, no iou
        
        iou_matrix = batch_iou(pred_bboxes, gt_bboxes)  # (M,N)

        cost_matrix = 1.0 - iou_matrix  # hungarian match on iou
        row_ind, col_ind = linear_sum_assignment(cost_matrix)

        # Compute average IoU over matched pairs 
        avg_iou = 0.0
        if len(row_ind) > 0:
            ious = [iou_matrix[i, j] for i, j in zip(row_ind, col_ind)]
            avg_iou = np.mean(ious)
    except Exception as e:
        print("Caught error in compute_hungarian_match_and_average_iou:", e)
        avg_iou = 0.0

    return avg_iou

image_name_to_baseline_avg_iou = {}
for idx, item in tqdm(enumerate(dataset)):
    image_name = item['id']

    gt_boxes = []
    gt_solution = json.loads(item['solution'])
    for sol in gt_solution:
        gt_boxes.append(sol['bbox_2d'])
    
    sam3_boxes = image_name_to_sam3_boxes[image_name]

    if sam3_boxes is None:
        image_name_to_baseline_avg_iou[image_name] = 0.0
        continue
    
    avg_iou = compute_hungarian_match_and_average_iou(np.array(sam3_boxes), np.array(gt_boxes))
    
    image_name_to_baseline_avg_iou[image_name] = avg_iou

117it [00:02, 57.02it/s]

7099it [02:06, 56.20it/s]


In [12]:
"""
{'id': 'id',
 'problem': "problem",
 'solution': '[{"bbox_2d": [35, 74, 585, 500], "point_2d": [295, 284]}]', -> need to load this and change to a scale of 1000 x 1000 (prediction_grid_size)
 'image': <PIL.PngImagePlugin.PngImageFile image mode=RGB size=840x840>, -> need to resize this to 1024 x 1024 (image_resize)
 'img_height': 640,
 'img_width': 478}
"""

for idx, item in tqdm(enumerate(dataset)):
    id_list.append(item['id'])

    problem_list.append(item['problem'])

    # img = cv2.resize(img, (image_resize, image_resize), interpolation=cv2.INTER_AREA)
  
    image_list.append(item['image'])
    
    img_height_list.append(item['img_height'])
    img_width_list.append(item['img_width'])
    
    # 840 because the image in the dataset is at 840 and the boxes are already resized to 840
    x_factor = prediction_grid_size / 840
    y_factor = prediction_grid_size / 840

    gt_boxes = []
    gt_points = []
    gt_solution = json.loads(item['solution'])
    for sol in gt_solution:
        gt_boxes.append(sol['bbox_2d'])
        gt_points.append(sol['point_2d'])

    solution = []

    for box_idx in range(len(gt_boxes)):
        solution.append({
            "bbox_2d": scale_box_coordinates(gt_boxes[box_idx], x_factor, y_factor), # [x1, y1, x2, y2]
            "point_2d": scale_point_coordinates(gt_points[box_idx], x_factor, y_factor) # [x, y]
            
        })

    solution_list.append(json.dumps(solution))

    baseline_iou = image_name_to_baseline_avg_iou[item['id']]
    baseline_iou_list.append(baseline_iou)

train_data = {
    'id': id_list,
    'problem': problem_list,
    'solution': solution_list,
    'image': image_list,
    'img_height': img_height_list,
    'img_width': img_width_list,
    'object_part': object_part_list,
    'object_hint_boxes': object_boxes_list,
    'baseline_iou': baseline_iou_list
}

7099it [02:12, 53.38it/s]


In [13]:
# the method resizes images to image_resize x image_resize and saves the dataset in HF format 
# 
dataset = create_local_dataset(
    train_data=train_data,
    output_dir=f"/data/VLMGroundingProject/Datasets/SegZeroVisualReasoner/VisionReasoner_part_qwen3",
    image_resize=image_resize
)

Saving the dataset (19/19 shards): 100%|██████████| 7099/7099 [00:07<00:00, 922.20 examples/s]

saved to: /data/VLMGroundingProject/Datasets/SegZeroVisualReasoner/VisionReasoner_part_qwen3
